# Column cards

A column card summarizes a column's inferred role, counts, statistics,
and value distribution. `report.columns` contains one dict per column.
Cards use every input row; relationship sampling does not affect them.

The fixture includes constant and all-null columns, repeated categories,
numeric measurements, dates, and invoice numbers with two formats.

In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(11)
n = 1500

base = pl.date_range(pl.date(2024, 1, 1), pl.date(2024, 1, 1) + pl.duration(weeks=n - 1),
                     interval="1w", eager=True)
plan = rng.choices(["starter", "growth", "enterprise"], weights=[70, 20, 10], k=n)
channel = ["web" if rng.random() < 0.995 else "app" for _ in range(n)]
unit = ["kg"] * n
legacy_flag = [None] * n
risk_score = [round(rng.uniform(0, 10), 1) for _ in range(n)]
email_domain = rng.choices(["gmail.com", "outlook.com", "corp.io", "proton.me"],
                           weights=[50, 25, 20, 5], k=n)
invoice_no = [f"INV-{i:04d}" if i % 2 == 0 else f"{i:04d}" for i in range(n)]
status_code = [rng.randrange(1, 7) for _ in range(n)]
is_active = [rng.random() < 0.8 for _ in range(n)]

df = pl.DataFrame({
    "signup_date": base,
    "plan": plan,
    "channel": channel,
    "unit": unit,
    "legacy_flag": legacy_flag,
    "risk_score": risk_score,
    "email_domain": email_domain,
    "invoice_no": invoice_no,
    "status_code": status_code,
    "is_active": is_active,
})

report = onceover.profile(df, budget="standard")
print(report.overview["headline"])

## A card at a glance

In [ ]:
import json

print(json.dumps(report.columns["risk_score"], indent=2))

`risk_score` has the inferred role `continuous`. Its `n`, `null_count`,
`distinct`, and `distinct_ratio` describe all input rows. `numeric_stats`
contains the mean, standard deviation, quartiles, zero and negative counts,
and whether every value is integral. `concentration` describes how often
values repeat; `top_values` lists the most frequent values.

The statistics block depends on dtype: strings have `string_stats`,
booleans have `bool_stats`, and dates have `temporal_stats`. The date
scan adds the most frequent interval to `temporal_stats`.
For `signup_date`, that interval is one week:

In [ ]:
print(json.dumps(report.columns["signup_date"]["temporal_stats"], indent=2))

## Roles

Role inference applies rules to dtype and exact counts. The resulting
label describes observed behavior and may differ from the column's meaning in the application:


In [ ]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Role': ['`all_null`', '`constant`', '`boolean`', '`categorical`', '`free_text`', '`identifier`', '`continuous`', '`date` / `datetime`', '`other`'], 'Inferred when': ['no non-null values', 'one distinct value', 'boolean dtype', 'strings with ≤ 500 distinct values; numeric with ≤ 12 distinct (n ≥ 30), or ≤ 100 distinct and ≤ 2% of rows (n ≥ 100)', 'strings with > 500 distinct values', 'numeric with distinct ratio ≥ 0.98', 'numeric, otherwise', 'date or datetime dtype', 'dtype the profiler treats as opaque'], 'Notes': ['flagged as a finding too', 'drop candidate', '', 'numeric categories carry a `treat_as_categorical` hint', 'excluded from nominal model passes', 'check intended meaning before treating as a join key', '', 'datetime analyzed at day granularity for gaps', 'reported in coverage']})
(GT(_doc_table)
 .fmt_markdown(columns=['Role', 'Inferred when', 'Notes'])
 .tab_header(title='Roles')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


A numeric measurement with no repeated values, such as random floats,
has a distinct ratio of 1.0 and receives the `identifier` role. That rule
cannot establish whether the column identifies an entity; check its
definition and use in the application. Date and datetime dtypes retain
their own roles. The numeric-categorical thresholds also need context:
a 13-level ordered code may still represent categories.

## Concentration

The `concentration` block describes the frequencies of non-null values.
The Herfindahl–Hirschman index is the sum of squared value shares: 1.0
for a constant column, or $1/k$ for $k$ equally frequent values.
Shannon entropy is reported in nats. Effective cardinality,
$\exp(\text{entropy})$, is the number of equally frequent values that
would have the same entropy. A six-level column dominated by one value
has effective cardinality near 1; a balanced six-level column has 6.
`top1_share` and `top3_share` give the fractions occupied by the most
frequent value and the three most frequent values.

The `email_domain` card illustrates these measures:


In [ ]:
print(json.dumps(report.columns["email_domain"]["concentration"], indent=2))
print(json.dumps(report.columns["email_domain"]["top_values"], indent=2))

## All cards at a glance

Render one row per column to compare roles, null fractions, and value
frequencies. The Great Tables styles below wrap long column names and roles:

In [ ]:
from great_tables import GT, loc, style

rows = []
for name, card in report.columns.items():
    conc = card.get("concentration")
    rows.append({
        "column": name,
        "dtype": card["dtype"],
        "role": card["role"],
        "nulls": f"{card['null_fraction']:.1%}",
        "distinct": card["distinct"],
        "top1 share": f"{conc['top1_share']:.1%}" if conc else "—",
        "eff. cardinality": f"{conc['effective_cardinality']:.2f}" if conc else "—",
    })

(GT(pl.DataFrame(rows))
 .tab_header(title="Column cards at a glance", subtitle="Every count is exact, over all rows")
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"),
            loc.body()))

The table shows `unit` as constant, `legacy_flag` as all null, `plan` and
`email_domain` as concentrated categoricals, `risk_score` as continuous,
and `invoice_no` as free text.

`invoice_no` has the `free_text` role because it exceeds the 500-value
threshold for string categories. It is also null-free, with 1,500 distinct
values in 1,500 rows, so its values can identify rows in this table.
The role excludes it from nominal model passes; uniqueness is checked
separately from its exact counts.

Cards are also the input to charts. The top values of a categorical column
make a quick bar chart — Altair over the card's `top_values`:

In [ ]:
import altair as alt

top = pl.DataFrame([
    {"value": t["value"], "share": t["share"]}
    for t in report.columns["email_domain"]["top_values"]
])

alt.Chart(top).mark_bar().encode(
    x=alt.X("share:Q", title="share of non-null values", axis=alt.Axis(format="%")),
    y=alt.Y("value:N", sort="-x"),
).properties(width=460, height=160)

## Format patterns

String cards carry one more block: `format_patterns`, the shape signatures
of the column's values. Digits become `#`, lowercase letters `a`, uppercase
letters `A`, other characters are kept, and signatures truncate at 24
characters. Two clusters of signatures in a column that should have one
can indicate mixed formats, as in this `invoice_no` fixture:

In [ ]:
print(json.dumps(report.columns["invoice_no"]["format_patterns"], indent=2))

Half the invoice numbers have the pattern `AAA-####` and half have
`####`. The card reports both signatures and their shares without listing
all 1,500 values. The mixed-format finding suggests normalization before
parsing; whether to change these values depends on the expected invoice
format.

## Card-level findings

The profiler also creates findings from card counts: constant and all-null
columns, nearly constant two-level columns such as `channel`, concentrated
values, numeric categories such as `status_code`, and identifier-like
columns. These receive lower selection priority than relationship findings.
The loop below prints the findings for `unit`, `legacy_flag`, `channel`,
and `status_code`, along with any suggested action.

In [ ]:
single = {"unit", "legacy_flag", "channel", "status_code"}
for f in report.findings:
    if len(f["subject"]["columns"]) == 1 and f["subject"]["columns"][0] in single:
        print(f"[{f['category']}] {f['claim']}")
        if f.get("interpretation"):
            print(f"    → {f['interpretation']}")

The thresholds for role rules, concentration flags, and format findings
are listed in
[](../methodology/column_cards.ipynb).